## mixture normal t and mad test

In [1]:
function run_ttest_summary(datasets)

    summary = Dict(
        "n" => Int[],
        "TP" => Float64[],
        "FP" => Float64[],
        "Power" => Float64[],
        "FDR" => Float64[]
    )

    # NEW: raw p-values for ranking metrics
    pvals_store =
        Dict{Tuple{Int,Int}, Vector{Float64}}()

    for n in n_values

        println("\nRunning t-test: n = $n")

        power_vals = Float64[]
        fdr_vals   = Float64[]
        tp_vals    = Float64[]
        fp_vals    = Float64[]

        for r in 1:R

            Y =
                datasets[(n,r)].Y

            is_DE =
                datasets[(n,r)].is_DE

            β_hat = [
                mean(Y[g])
                for g in 1:G
            ]

            s_hat = [
                std(Y[g], corrected = true)
                for g in 1:G
            ]

            t_stats =
                sqrt(n) .* β_hat ./ s_hat

            tdist =
                TDist(n - 1)

            # raw p-values
            pvals =
                2 .* ccdf.(
                    Ref(tdist),
                    abs.(t_stats)
                )

            # -------------------------------------------------
            # NEW: save raw p-values for this replicate
            # -------------------------------------------------
            pvals_store[(n,r)] =
                copy(pvals)

            adj_p =
                adjust(
                    pvals,
                    BenjaminiHochberg()
                )

            reject =
                adj_p .<= 0.05

            TP =
                sum(reject .& is_DE)

            FP =
                sum(reject .& .!is_DE)

            power =
                TP / max(sum(is_DE), 1)

            fdp =
                FP / max(sum(reject), 1)

            push!(tp_vals, TP)
            push!(fp_vals, FP)
            push!(power_vals, power)
            push!(fdr_vals, fdp)
        end

        push!(summary["n"], n)
        push!(summary["TP"], mean(tp_vals))
        push!(summary["FP"], mean(fp_vals))
        push!(summary["Power"], mean(power_vals))
        push!(summary["FDR"], mean(fdr_vals))

        println(
            "n = $n → ",
            "Power = $(round(mean(power_vals), digits=4)), ",
            "FDR = $(round(mean(fdr_vals), digits=4))"
        )
    end

    return summary, pvals_store
end

run_ttest_summary (generic function with 1 method)

In [4]:
function extract_is_DE_store(datasets)

    is_DE_store =
        Dict{Tuple{Int,Int}, Vector{Bool}}()

    for n in n_values
        for r in 1:R
            is_DE_store[(n,r)] =
                copy(datasets[(n,r)].is_DE)
        end
    end

    return is_DE_store
end

extract_is_DE_store (generic function with 1 method)

In [5]:
using Distributions
using Random
using Statistics
using MultipleTesting
using JLD2


# =========================================================
# 1. Simulation parameters
# =========================================================

n_values = [4, 5, 6, 8, 10, 20, 30, 40]

G = 1000
R = 100

p_signal = 0.1
β_signal = 4.0

s0_sq_true = 1.0
d0_true    = 4.0

tau_mix = 9.0

eps_values = [0.05, 0.10, 0.15]


# =========================================================
# 2. Draw mixture-Normal errors
#
# baseline component:
#     Normal(0, 1)       -> variance 1
#
# contamination component:
#     Normal(0, tau)     -> variance tau^2
# =========================================================

function rand_mixture_normal_errors(
    rng::AbstractRNG,
    n::Int;
    eps::Float64,
    tau::Float64
)

    z = Vector{Float64}(undef, n)

    @inbounds for j in 1:n

        if rand(rng) < eps

            z[j] =
                rand(
                    rng,
                    Normal(0.0, tau)
                )

        else

            z[j] =
                rand(
                    rng,
                    Normal(0.0, 1.0)
                )

        end
    end

    return z
end


# =========================================================
# 3. Generate datasets for one epsilon
# =========================================================

function generate_normal_mixture_datasets(
    eps_mix::Float64
)

    datasets =
        Dict{Tuple{Int,Int}, Any}()

    for n in n_values
        for r in 1:R

            # Same seed across epsilon settings
            # so DE indicators and sigma are matched
            rng =
                MersenneTwister(
                    10_000 + 100n + r
                )


            # -------------------------------------------------
            # DE indicators
            # -------------------------------------------------

            is_DE =
                rand(rng, G) .< p_signal


            # -------------------------------------------------
            # Gene-specific variances
            # -------------------------------------------------

            σ2 =
                rand(
                    rng,
                    InverseGamma(
                        d0_true / 2,
                        d0_true * s0_sq_true / 2
                    ),
                    G
                )

            σ =
                sqrt.(σ2)


            # -------------------------------------------------
            # Generate observations
            # -------------------------------------------------

            Y =
                Vector{Vector{Float64}}(
                    undef,
                    G
                )

            @inbounds for g in 1:G

                # Standardized signal fixed across n
                βg =
                    is_DE[g] ?
                    β_signal * σ[g] / sqrt(n) :
                    0.0

                z =
                    rand_mixture_normal_errors(
                        rng,
                        n;
                        eps = eps_mix,
                        tau = tau_mix
                    )

                Y[g] =
                    βg .+ σ[g] .* z
            end


            datasets[(n,r)] = (
                Y = Y,
                is_DE = is_DE,
                σ = σ,
                σ2 = σ2,
                error_dist = "Normal Mixture",
                eps_mix = eps_mix,
                tau_mix = tau_mix,
                β_signal = β_signal,
                signal_scaled_by_sqrt_n = true,
                s0_sq_true = s0_sq_true,
                d0_true = d0_true
            )
        end
    end

    return datasets
end


# =========================================================
# 4. Run t-test for all epsilon settings
# =========================================================

for eps_mix in eps_values

    println("\n====================================")
    println("Running mixture Normal eps = $eps_mix")
    println("====================================")


    # -----------------------------------------------------
    # 1. Generate data
    # -----------------------------------------------------

    datasets =
        generate_normal_mixture_datasets(
            eps_mix
        )


    # -----------------------------------------------------
    # 2. Store truth labels
    # -----------------------------------------------------

    is_DE_store =
        extract_is_DE_store(
            datasets
        )


    # -----------------------------------------------------
    # 3. Run ordinary t-test
    # -----------------------------------------------------

    ttest_summary,
    ttest_pvals_store =
        run_ttest_summary(
            datasets
        )


    # -----------------------------------------------------
    # 4. Save simulation settings
    # -----------------------------------------------------

    simulation_params = (
        n_values = n_values,
        G = G,
        R = R,
        p_signal = p_signal,
        β_signal = β_signal,
        s0_sq_true = s0_sq_true,
        d0_true = d0_true,
        error_dist = "Normal Mixture",
        eps_mix = eps_mix,
        tau_mix = tau_mix,
        signal_scaled_by_sqrt_n = true
    )


    # -----------------------------------------------------
    # 5. Make filename
    #
    # 0.05 -> 0p05
    # 0.10 -> 0p1
    # 0.15 -> 0p15
    # -----------------------------------------------------

    eps_label =
        replace(
            string(eps_mix),
            "." => "p"
        )

    filename =
        "ttest_pvalues_normal_mixture_eps$(eps_label).jld2"


    # -----------------------------------------------------
    # 6. Save
    # -----------------------------------------------------

    @save filename simulation_params ttest_summary ttest_pvals_store is_DE_store

    println("Saved: $filename")

end


Running mixture Normal eps = 0.05

Running t-test: n = 4
n = 4 → Power = 0.0029, FDR = 0.0483

Running t-test: n = 5
n = 5 → Power = 0.0097, FDR = 0.0633

Running t-test: n = 6
n = 6 → Power = 0.0398, FDR = 0.0333

Running t-test: n = 8
n = 8 → Power = 0.184, FDR = 0.0242

Running t-test: n = 10
n = 10 → Power = 0.2596, FDR = 0.0343

Running t-test: n = 20
n = 20 → Power = 0.3283, FDR = 0.0217

Running t-test: n = 30
n = 30 → Power = 0.2901, FDR = 0.0154

Running t-test: n = 40
n = 40 → Power = 0.2431, FDR = 0.0096
Saved: ttest_pvalues_normal_mixture_eps0p05.jld2

Running mixture Normal eps = 0.1

Running t-test: n = 4
n = 4 → Power = 0.0019, FDR = 0.05

Running t-test: n = 5
n = 5 → Power = 0.0052, FDR = 0.0512

Running t-test: n = 6
n = 6 → Power = 0.0182, FDR = 0.0217

Running t-test: n = 8
n = 8 → Power = 0.0858, FDR = 0.014

Running t-test: n = 10
n = 10 → Power = 0.1209, FDR = 0.0269

Running t-test: n = 20
n = 20 → Power = 0.1187, FDR = 0.0086

Running t-test: n = 30
n = 30 → P

In [6]:
using FastGaussQuadrature
using Distributions
using StatsFuns: logsumexp
using Optim
using MultipleTesting
using Statistics, Random, LinearAlgebra
using ApproxFun, SpecialFunctions

# =========================================================
# 1. Exact MAD-about-mean likelihood
# =========================================================

struct MADDistribution{T<:Real,S<:Integer} <: ContinuousUnivariateDistribution
    σ::T
    n::S
end

import Distributions: pdf, logpdf, insupport, minimum, maximum

function G_recursive(max_r::Int; a=0.0, b=100)
    d = Interval(a, b)
    G = Vector{Fun}(undef, max_r+1)
    G[1] = Fun(x -> 1.0, d)
    for r in 1:max_r
        integrand = Fun(x -> exp(-(x^2)/(2r*(r+1))), d) * G[r]
        G[r+1] = cumsum(integrand)
    end
    return G
end

const _Gcache = Dict{Int,Vector{Fun}}()
_getG(n) = get!(_Gcache, n) do
    G_recursive(n)
end

function _pdf_sigma1(n::Int, m::Real)
    m < 0 && return zero(float(m))
    z   = n * m / 2
    cst = n^(3/2) / (2^((n+1)/2) * π^((n-1)/2))
    G   = _getG(n)
    s   = 0.0
    @inbounds for k in 1:(n-1)
        s += binomial(n, k) * exp(-(m^2 * n^3) / (8k*(n-k))) * G[k](z) * G[n-k](z)
    end
    return cst * s
end

insupport(::MADDistribution, x::Real) = x ≥ 0
minimum(::MADDistribution) = 0.0
maximum(::MADDistribution) = Inf

pdf(d::MADDistribution, m::Real) =
    m < 0 ? 0.0 : (1 / d.σ) * _pdf_sigma1(d.n, m / d.σ)

function logpdf(d::MADDistribution, m::Real)
    m < 0 && return -Inf
    σ, n = d.σ, d.n
    z    = n * m / (2σ)
    logC = (3/2)*log(n) - ((n+1)/2)*log(2) - ((n-1)/2)*log(π)
    G    = _getG(n)

    logs = Float64[]
    @inbounds for k in 1:(n-1)
        val1 = G[k](z)
        val2 = G[n-k](z)
        if val1 > 0 && val2 > 0
            push!(logs,
                log(binomial(n, k)) -
                (m^2 * n^3) / (8σ^2 * k*(n-k)) +
                log(val1) + log(val2)
            )
        end
    end

    return -log(σ) + logC + (isempty(logs) ? -Inf : logsumexp(logs))
end


logpdf (generic function with 83 methods)

In [7]:
using FastGaussQuadrature
using Distributions
using Statistics
using MultipleTesting

# =========================================================
# Precompute quadrature for the exact MAD-normalized null
#
# Integral:
#
# p(t) = ∫ 2 Φbar(|t| c_n w) f_D(w | sigma=1,n) dw
#
# Transform:
#     w = u / (1-u),   u ∈ (0,1)
# =========================================================

function build_mad_null_quadrature(
    n::Int;
    K::Int = 400
)

    # Gauss-Legendre on [-1,1]
    x, qweights = gausslegendre(K)

    # Map to (0,1)
    u = (x .+ 1) ./ 2
    qweights ./= 2

    # Transform (0,1) -> (0,Inf)
    w_nodes = u ./ (1 .- u)

    jac = 1 ./ (1 .- u).^2

    # Exact MAD density evaluated ONLY ONCE
    mad_density = [
        pdf(
            MADDistribution(1.0, n),
            w
        )
        for w in w_nodes
    ]

    # Complete integration weights
    weights =
        qweights .* jac .* mad_density

    # Optional normalization to remove tiny numerical integration error
    weights ./= sum(weights)

    c_n =
        sqrt(pi / 2) *
        sqrt(n / (n - 1))

    return (
        w = w_nodes,
        weights = weights,
        c_n = c_n
    )
end

function mad_normalized_pvalue_fast(
    t_obs::Real,
    quad
)

    x = abs(t_obs)

    tails =
        2 .* ccdf.(
            Normal(),
            x .* quad.c_n .* quad.w
        )

    p =
        dot(
            quad.weights,
            tails
        )

    return clamp(p, 0.0, 1.0)
end

mad_quad_store =
    Dict{Int, Any}()

for n in n_values

    println(
        "Precomputing MAD null quadrature for n = $n"
    )

    @time mad_quad_store[n] =
        build_mad_null_quadrature(
            n;
            K = 400
        )
end

Precomputing MAD null quadrature for n = 4
  2.512597 seconds (30.04 M allocations: 1.524 GiB, 8.39% gc time, 99.47% compilation time)
Precomputing MAD null quadrature for n = 5
  0.004522 seconds (13.42 k allocations: 901.188 KiB)
Precomputing MAD null quadrature for n = 6
  0.006100 seconds (16.33 k allocations: 990.672 KiB)
Precomputing MAD null quadrature for n = 8
  0.007450 seconds (22.11 k allocations: 1.271 MiB)
Precomputing MAD null quadrature for n = 10
  0.007308 seconds (27.86 k allocations: 1.615 MiB)
Precomputing MAD null quadrature for n = 20
  0.010271 seconds (62.64 k allocations: 3.528 MiB)
Precomputing MAD null quadrature for n = 30
  0.014426 seconds (95.34 k allocations: 5.621 MiB)
Precomputing MAD null quadrature for n = 40
  0.017082 seconds (128.82 k allocations: 7.679 MiB)


In [8]:
function run_mad_test_summary(
    datasets
)

    summary = Dict(
        "n" => Int[],
        "TP" => Float64[],
        "FP" => Float64[],
        "Power" => Float64[],
        "FDR" => Float64[]
    )

    # NEW: store raw MAD-test p-values for ranking metrics
    pvals_store =
        Dict{Tuple{Int,Int}, Vector{Float64}}()

    for n in n_values

        println(
            "\nRunning MAD test: n = $n"
        )

        quad =
            mad_quad_store[n]

        power_vals = Float64[]
        fdr_vals   = Float64[]
        tp_vals    = Float64[]
        fp_vals    = Float64[]

        for r in 1:R

            Y =
                datasets[(n,r)].Y

            is_DE =
                datasets[(n,r)].is_DE

            # -------------------------------------------------
            # Sample means
            # -------------------------------------------------

            β_hat = [
                mean(Y[g])
                for g in 1:G
            ]

            # -------------------------------------------------
            # MAD about sample mean
            # -------------------------------------------------

            mad_obs = [
                mean(
                    abs.(
                        Y[g] .- β_hat[g]
                    )
                )
                for g in 1:G
            ]

            # -------------------------------------------------
            # Normal-theory corrected MAD estimate of sigma
            # -------------------------------------------------

            sigma_hat_mad =
                quad.c_n .* mad_obs

            # -------------------------------------------------
            # MAD-normalized statistic
            # -------------------------------------------------

            t_mad =
                sqrt(n) .* β_hat ./
                sigma_hat_mad

            # -------------------------------------------------
            # Gaussian-null calibrated RAW p-values
            # -------------------------------------------------

            pvals = [
                mad_normalized_pvalue_fast(
                    t,
                    quad
                )
                for t in t_mad
            ]

            # -------------------------------------------------
            # NEW: save raw p-values
            # -------------------------------------------------

            pvals_store[(n,r)] =
                copy(pvals)

            # -------------------------------------------------
            # BH adjustment
            # -------------------------------------------------

            adj_p =
                adjust(
                    pvals,
                    BenjaminiHochberg()
                )

            reject =
                adj_p .<= 0.05

            # -------------------------------------------------
            # Metrics
            # -------------------------------------------------

            TP =
                sum(reject .& is_DE)

            FP =
                sum(reject .& .!is_DE)

            power =
                TP / max(sum(is_DE), 1)

            fdp =
                FP / max(sum(reject), 1)

            push!(tp_vals, TP)
            push!(fp_vals, FP)
            push!(power_vals, power)
            push!(fdr_vals, fdp)
        end

        push!(summary["n"], n)
        push!(summary["TP"], mean(tp_vals))
        push!(summary["FP"], mean(fp_vals))
        push!(summary["Power"], mean(power_vals))
        push!(summary["FDR"], mean(fdr_vals))

        println(
            "n = $n → ",
            "Power = $(round(mean(power_vals),digits=4)), ",
            "FDR = $(round(mean(fdr_vals),digits=4))"
        )
    end

    return summary, pvals_store
end

run_mad_test_summary (generic function with 1 method)

In [10]:
nonEB_normal_results =
    Dict{Float64, Any}()

for eps in eps_values

    println(
        "\n\n",
        "==============================================\n",
        "       Normal mixture: epsilon = $eps\n",
        "=============================================="
    )

    # -----------------------------------------------------
    # Generate datasets
    # -----------------------------------------------------

    datasets_eps =
        generate_normal_mixture_datasets(
            eps
        )

    # -----------------------------------------------------
    # Store truth labels for ranking
    # -----------------------------------------------------

    is_DE_store =
        Dict{Tuple{Int,Int}, Vector{Bool}}()

    for n in n_values
        for r in 1:R
            is_DE_store[(n,r)] =
                copy(
                    datasets_eps[(n,r)].is_DE
                )
        end
    end

    # -----------------------------------------------------
    # Ordinary t-test
    # -----------------------------------------------------

    ttest_summary,
    ttest_pvals_store =
        run_ttest_summary(
            datasets_eps
        )

    # -----------------------------------------------------
    # MAD-normalized test
    # -----------------------------------------------------

    mad_test_summary,
    mad_test_pvals_store =
        run_mad_test_summary(
            datasets_eps
        )

    # -----------------------------------------------------
    # Store in memory
    # -----------------------------------------------------

    nonEB_normal_results[eps] = (
        ttest = ttest_summary,
        mad_test = mad_test_summary,
        ttest_pvals_store = ttest_pvals_store,
        mad_test_pvals_store = mad_test_pvals_store,
        is_DE_store = is_DE_store
    )

    # -----------------------------------------------------
    # Simulation settings
    # -----------------------------------------------------

    simulation_params = (
        n_values = n_values,
        G = G,
        R = R,
        p_signal = p_signal,
        β_signal = β_signal,
        s0_sq_true = s0_sq_true,
        d0_true = d0_true,
        error_dist = "Normal Mixture",
        eps_mix = eps,
        tau_mix = tau_mix,
        signal_scaled_by_sqrt_n = true
    )

    # -----------------------------------------------------
    # File name
    # -----------------------------------------------------

    eps_label =
        replace(
            string(eps),
            "." => "p"
        )

    filename =
        "nonEB_normal_mixture_eps$(eps_label).jld2"

    # -----------------------------------------------------
    # Save
    # -----------------------------------------------------

    @save filename simulation_params ttest_summary mad_test_summary ttest_pvals_store mad_test_pvals_store is_DE_store

    println(
        "Saved: $filename"
    )
end



       Normal mixture: epsilon = 0.05

Running t-test: n = 4
n = 4 → Power = 0.0029, FDR = 0.0483

Running t-test: n = 5
n = 5 → Power = 0.0097, FDR = 0.0633

Running t-test: n = 6
n = 6 → Power = 0.0398, FDR = 0.0333

Running t-test: n = 8
n = 8 → Power = 0.184, FDR = 0.0242

Running t-test: n = 10
n = 10 → Power = 0.2596, FDR = 0.0343

Running t-test: n = 20
n = 20 → Power = 0.3283, FDR = 0.0217

Running t-test: n = 30
n = 30 → Power = 0.2901, FDR = 0.0154

Running t-test: n = 40
n = 40 → Power = 0.2431, FDR = 0.0096

Running MAD test: n = 4
n = 4 → Power = 0.0028, FDR = 0.0433

Running MAD test: n = 5
n = 5 → Power = 0.0121, FDR = 0.0516

Running MAD test: n = 6
n = 6 → Power = 0.0352, FDR = 0.035

Running MAD test: n = 8
n = 8 → Power = 0.1744, FDR = 0.0308

Running MAD test: n = 10
n = 10 → Power = 0.2535, FDR = 0.0381

Running MAD test: n = 20
n = 20 → Power = 0.4387, FDR = 0.03

Running MAD test: n = 30
n = 30 → Power = 0.4841, FDR = 0.0545

Running MAD test: n = 40
n = 40 → P